# Units and Quantities in CIM

In CIM, a physical quantity such as `Voltage`, `ActivePower` or `Length` is a **datatype**: a value with a unit symbol (`UnitSymbol`, e.g. `V`) and a unit multiplier (`UnitMultiplier`, e.g. `k`). The CIM-Graph profiles in the [`cimprofiles`](https://github.com/PNNL-CIM-Tools/CIM-Profiles) package define each datatype as a subclass of `CIMUnit` (`cimprofiles.units`), backed by the [Pint](https://pint.readthedocs.io/en/stable/) units library.

A `CIMUnit` stores its value in its datatype's unit. That is the SI base unit (V, W, VAr, VA, ohm, S, m, ...) for almost every datatype; a few fix a multiplier, such as `KiloActivePower` (kW).

The rules in short:

- **Importing** data: give the source unit, `cim.ActivePower(5.5, 'MW')`.
- **Exporting** data: convert with `.to()`, `load.p.to('MW')`, which returns a float.
- Never scale by `1e3` or `1e6` by hand.
- Compare with a number through `.to()` or `float()`, never the `CIMUnit` itself.

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim
from cimprofiles.units import CIMUnit

## Creating quantities

The constructor takes a value and the unit it is in (`input_unit`, default: the datatype's unit). The unit can also be split into a CIM multiplier and symbol, and the value can be a Pint `Quantity` or another `CIMUnit`:

In [2]:
voltage = cim.Voltage(12.47, 'kV')
print(voltage)                                                 # stored in volts
print(cim.Voltage(12470))                                      # no unit: the datatype's unit
print(cim.Voltage(12.47, input_unit='V', input_multiplier='k'))
print(cim.Voltage(voltage.quantity))                           # from a Pint Quantity
print(cim.Length(1, 'mi'), cim.ReactivePower(2, 'MVAr'), cim.ApparentPower(50, 'MVA'))
print(voltage.unit, voltage.multiplier, isinstance(voltage, CIMUnit))

12470.0 volt
12470.0 volt
12470.0 volt
12470.0 volt
1609.344 meter 2000000.0 VAr 50000000.0 volt_ampere
UnitSymbol.V UnitMultiplier.none True


## Reading values

`.to(unit)` returns the value as a float in any unit of the same dimension. `float()` and `.value` give the stored value, and `.quantity` is the Pint `Quantity`:

In [3]:
print(voltage.to('kV'), voltage.to('V'), voltage.to('MV'))
print(float(voltage), voltage.value)
print(repr(voltage.quantity))

12.47 12470.0 0.01247
12470.0 12470.0
<Quantity(12470.0, 'volt')>


### Comparing with numbers

A `CIMUnit` equals another `CIMUnit` of the same value, but **never compare it with a raw number**. Ordering (`voltage < 13000`) and arithmetic (`voltage + 1`) raise `TypeError`, and so does `voltage == 12470.0` from cim-profiles 0.2.0a7 (before that it is silently `False`). Convert first:

In [4]:
print(voltage == cim.Voltage(12470))    # same datatype and value
print(voltage.to('kV') > 12)            # compare in a stated unit
print(float(voltage) == 12470.0)        # or in the stored unit

True
True
True


## Units on CIM objects

CIM attributes take quantities of their datatype. When `CIMG_USE_UNITS` is `true` (the default), the file and database readers also create `CIMUnit` values, so attributes read from a model behave the same way.

In [5]:
line = cim.ACLineSegment(
    name='line_1',
    r=cim.Resistance(0.185, 'ohm'),
    x=cim.Reactance(0.382, 'ohm'),
    length=cim.Length(2.5, 'km'),
)
load = cim.EnergyConsumer(name='load_1', p=cim.ActivePower(1.5, 'MW'), q=cim.ReactivePower(500, 'kVAr'))

print(f'Line: r = {line.r.to("ohm")} ohm, length = {line.length.to("km")} km = {line.length.to("mi"):.3f} mi')
print(f'Load: p = {load.p.to("kW")} kW, q = {load.q.to("MVAr")} MVAr')

Line: r = 0.185 ohm, length = 2.5 km = 1.553 mi
Load: p = 1500.0 kW, q = 0.5 MVAr


## Calculations

`CIMUnit` has no arithmetic operators. Calculate with the Pint quantities (`.quantity`), which check dimensions, and wrap the result in the datatype it should be. A result of the wrong dimension raises Pint's `DimensionalityError`.

Three-phase rated current, I = S / (√3 · V):

In [6]:
import math

rated_s = cim.ApparentPower(50, 'MVA')
rated_u = cim.Voltage(138, 'kV')
rated_i = cim.CurrentFlow(rated_s.quantity / (math.sqrt(3) * rated_u.quantity))
print(f'Rated current: {rated_i.to("A"):.1f} A')

Rated current: 209.2 A


Per-unit impedances (as in MATPOWER or PSS/E data) convert to ohms through the base impedance, Z_base = V_base² / S_base:

In [7]:
base_power = cim.ApparentPower(100, 'MVA')
base_impedance = cim.Impedance(rated_u.quantity ** 2 / base_power.quantity)

r_pu = 0.01
r = cim.Resistance(r_pu * base_impedance.to('ohm'), 'ohm')
print(f'Z_base = {base_impedance.to("ohm"):.2f} ohm, r = {r_pu} pu = {r.to("ohm"):.4f} ohm')
print(f'Back to per unit: {r.to("ohm") / base_impedance.to("ohm"):.4f} pu')

Z_base = 190.44 ohm, r = 0.01 pu = 1.9044 ohm
Back to per unit: 0.0100 pu


Other quantities follow the same pattern, such as per-length line constants and energy:

In [8]:
r_per_length = cim.ResistancePerLength(line.r.quantity / line.length.quantity)
print(f'r = {r_per_length.to("ohm/km"):.4f} ohm/km')

energy = cim.RealEnergy(load.p.quantity * cim.Hours(24, 'h').quantity)
print(f'Energy in a day: {energy.to("MWh"):.1f} MWh')

r = 0.0740 ohm/km
Energy in a day: 36.0 MWh


## Units in files

### CIM XML

CIM XML values are plain numbers in the datatype's unit (base SI for the datatypes above), with no unit labels: the profile defines each attribute's unit. The readers read them back in that unit. The writer adds no `rdf:datatype` to them.

A small model to write:

In [9]:
import tempfile

from cimgraph.models import FeederModel
from cimgraph.utils import write_json_ld, write_xml

network = FeederModel(connection=None, container=cim.Feeder(name='demo'), distributed=False)
network.create(cim.EnergyConsumer, name='load_1', p=cim.ActivePower(150, 'kW'), q=cim.ReactivePower(50, 'kVAr'))
network.create(cim.OverheadWireInfo, name='wire_1', radius=cim.Length(5, 'mm'))

output_dir = tempfile.mkdtemp()
xml_file = os.path.join(output_dir, 'demo.xml')
write_xml(network, xml_file)
for row in open(xml_file):
    if 'EnergyConsumer.p' in row or 'EnergyConsumer.q' in row or 'WireInfo.radius' in row:
        print(row.strip())

<cim:EnergyConsumer.p>150000.0</cim:EnergyConsumer.p>
<cim:EnergyConsumer.q>50000.0</cim:EnergyConsumer.q>
<cim:WireInfo.radius>0.005</cim:WireInfo.radius>


### JSON-LD units conventions

JSON-LD files are written in the units of a **units convention**: a publishing convention that each profile defines, per quantity kind and per property, so a file can use the units its community expects. The convention's units vocabulary is named in the file header's `dct:conformsTo`, and `JSONLDFile` converts the numbers back when it reads the file, so in Python the values are the same `CIMUnit`s whichever convention was used.

- The default is the profile's default convention: base SI for most profiles (`si`), and the CGMES units (MW, kV, km, ...) for `cgmes_3_0_0`.
- `write_json_ld(..., convention='distribution')` (`cimhub_2023`, `cimhub_2026`, `cim18v17_distribution`) writes kW, kVAr, kVA and kWh, line constants per km (ohm/km, uS/km), and wire and cable sizes in mm.
- `cim18v17_transmission` has a `transmission` convention (MW and kV as in CGMES 3.0, line lengths in km and line constants per km).

In [10]:
import json

def show_units(filename):
    document = json.load(open(filename))
    print('dct:conformsTo:', document['dct:conformsTo'])
    for node in document['@graph']:
        for key, value in node.items():
            if key in ('cim:EnergyConsumer.p', 'cim:EnergyConsumer.q', 'cim:WireInfo.radius'):
                print(f'  {key}: {value}')

json_file = os.path.join(output_dir, 'demo.jsonld')
for convention in [None, 'distribution']:
    write_json_ld(network, json_file, convention=convention)
    print(f'convention={convention!r}')
    show_units(json_file)

convention=None
dct:conformsTo: ['https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-si.jsonld']
  cim:EnergyConsumer.p: 150000
  cim:EnergyConsumer.q: 50000
  cim:WireInfo.radius: 0.005
convention='distribution'
dct:conformsTo: ['https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-distribution.jsonld']
  cim:EnergyConsumer.p: 150
  cim:EnergyConsumer.q: 50
  cim:WireInfo.radius: 5


Reading the `distribution` file back gives the same quantities:

In [11]:
from cimgraph.databases import JSONLDFile

read_back = FeederModel(connection=JSONLDFile(filename=json_file), container=cim.Feeder(), distributed=False)
load_1 = read_back.first(cim.EnergyConsumer)
wire_1 = read_back.first(cim.OverheadWireInfo)
print(f'p = {load_1.p} = {load_1.p.to("kW")} kW')
print(f'radius = {wire_1.radius} = {wire_1.radius.to("mm")} mm')

p = 150000.0 watt = 150.0 kW
radius = 0.005 meter = 5.0 mm


## Custom power system units

Pint's registry is extended with power engineering units in `cimprofiles/units/cim_units/units.txt`, such as `VAr` and `VArh` for reactive power and energy, `VA` and `VAh` for apparent power and energy, and ratios such as `VPerV`. Reactive units carry a dimensionless `imaginary` factor, so `VAr` converts like watts in calculations (e.g. S = √(P² + Q²)).

## Summary

- CIM quantities are `CIMUnit` subclasses (`cimprofiles.units`) that store values in their datatype's unit, almost always base SI.
- Create them with the source unit, `cim.ActivePower(5.5, 'MW')`; read them with `.to('MW')`; compare through `.to()` or `float()`.
- Calculate with `.quantity` and wrap the result in its datatype.
- CIM XML holds plain numbers in base SI units. JSON-LD uses a units convention (`write_json_ld(..., convention=...)`), named in its header and converted back on read.